In [239]:
# Code to map Maven elements to Origami's Incidents elements
# Using semantic similarity library
# Author: Sandeep Chintabathina
# April 2026

In [240]:
# For each Maven database data element, check if there is an exact or close matching data element in the origami Incidents package
# The output will list the maven element alongside the origami element and other supporting attributes

In [241]:
from sentence_transformers import SentenceTransformer


In [242]:
import pandas as pd
import numpy as np

In [243]:
# Read maven elements
df_maven = pd.read_csv('output/all_db_elements_use_this.csv',na_filter=False)
len(df_maven)

6232

In [244]:
df_maven.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6232 entries, 0 to 6231
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   QUESTION_ID     6232 non-null   object
 1   value           6232 non-null   object
 2   QUESTIONSET_ID  6232 non-null   object
 3   MODEL_NAME      6232 non-null   object
dtypes: object(4)
memory usage: 194.9+ KB


In [245]:
def clean_up(name):
    new_name =''
    for ch in name:
        # If space, digits, uppercase letters, lowercase letters, or underscore, add it to new name otherwise ignore
        if (ord(ch)==32 or 48<=ord(ch)<=57 or 65<=ord(ch)<=90 or 97<=ord(ch)<=122 or ord(ch)==95 ):
            new_name+=ch
    return new_name.strip()

In [246]:
clean_up('[sdfs] 34_AS ^#~')

'sdfs 34_AS'

In [247]:
# Clean up questions or data elements
df_maven.loc[:,'QUESTION_ID'] =[ clean_up(q) for q in df_maven.loc[:,'QUESTION_ID']]

In [248]:
df_maven.loc[:,'MODEL_NAME'].value_counts(dropna=False)

MODEL_NAME
DiseaseSurveillanceModel_General        3192
DiseaseSurveillanceModel_STD            1361
DiseaseSurveillanceModel_Hep             754
Childhood_Lead_Child_Model               386
ClusterModel                             205
Childhood_Lead_Investigation_Model       136
DiseaseSurveillanceModel_Coinfection     114
PortalApplicationModel                    41
EHRAndScreeningModel_STD                  22
DiseaseSurveillanceModel_AggNETSS         14
IsolationAndControlModel                   7
Name: count, dtype: int64

In [305]:
# General clean up
df_maven.loc[:,'MODEL_NAME'] = [m.strip() for m in df_maven.loc[:,'MODEL_NAME']]

In [250]:
# creating a sorting order for model name which is then used to sort the dataframe 
# Using the order from value counts above
sorter = ['DiseaseSurveillanceModel_General','DiseaseSurveillanceModel_STD','DiseaseSurveillanceModel_Hep','Childhood_Lead_Child_Model',
          'ClusterModel','Childhood_Lead_Investigation_Model','DiseaseSurveillanceModel_Coinfection','PortalApplicationModel',
          'EHRAndScreeningModel_STD','DiseaseSurveillanceModel_AggNETSS','IsolationAndControlModel']

In [251]:
# convert model_name column to category
df_maven.MODEL_NAME = df_maven.MODEL_NAME.astype('category')
# Set sorter as the categories hierarchy
df_maven.MODEL_NAME = df_maven.MODEL_NAME.cat.set_categories(sorter)


In [252]:
df_maven.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6232 entries, 0 to 6231
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   QUESTION_ID     6232 non-null   object  
 1   value           6232 non-null   object  
 2   QUESTIONSET_ID  6232 non-null   object  
 3   MODEL_NAME      6232 non-null   category
dtypes: category(1), object(3)
memory usage: 152.7+ KB


In [253]:
# Sort df on model_name
df_maven = df_maven.sort_values(['MODEL_NAME'])

In [254]:
df_maven

,QUESTION_ID,value,QUESTIONSET_ID,MODEL_NAME
3115,CURRENT_TELEPHONE_WORK,(999) 999-9999,DEMOGRAPHIC,DiseaseSurveillanceModel_General
2691,EAT_YOGURT_PURCHASED_FROM,Yogurt Melts,RISK_HISTORY,DiseaseSurveillanceModel_General
2692,CDR_PERSON_OR_AGENCY_REPORTING,Zyan Paresa,PORTAL_CDR_CASE_SUBMISSION,DiseaseSurveillanceModel_General
2693,ANTACID_EXPOSURE,UNKNOWN,CLINICAL,DiseaseSurveillanceModel_General
2694,DIARY_DAY_1_DINNER,Zippys -No Bean Chili,RISK_HISTORY,DiseaseSurveillanceModel_General
...,...,...,...,...
6185,PROGRAM_CODE,STD,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel
6186,PRODUCT_DESCRIPTION,YERSINIOSIS; Yersinia enterocolitica; psudotob...,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel
6190,REPORT_CATEGORY_LAB,Urgent - upon order,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel
6189,REPORT_CATEGORY_PROVIDER,Urgent,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel


In [255]:
df_maven = df_maven.reset_index(drop=True)

In [256]:
# duplicate question ids
df_maven[df_maven.duplicated(['QUESTION_ID'])]

,QUESTION_ID,value,QUESTIONSET_ID,MODEL_NAME
3207,RACE_NATIVE_HAWAIIAN_PACIFIC_ISLANDER_DETAIL,2500-7,DEMOGRAPHIC,DiseaseSurveillanceModel_STD
3218,CDR_CONTACT_TYPE_RELATION_OTHER_DESC,Boyfriend/Babies Daddy,PORTAL_CDR_CASE_SUBMISSION,DiseaseSurveillanceModel_STD
3249,CDR_DATA_SUBMITTED_DATE,12/28/2022,PORTAL_CDR_CASE_SUBMISSION,DiseaseSurveillanceModel_STD
3267,ADDRESS_TYPE,3,REPORTING_MORBIDITY,DiseaseSurveillanceModel_STD
3305,RACE_REFUSED,NO,DEMOGRAPHIC,DiseaseSurveillanceModel_STD
...,...,...,...,...
6222,EPISODE_DATE,12/31/2016,ADMINISTRATIVE,DiseaseSurveillanceModel_AggNETSS
6224,MMWR_WEEK,53,ADMINISTRATIVE,DiseaseSurveillanceModel_AggNETSS
6226,ISO_QUAR_CONTACT_INSTRUCTIONS,"Surveillance, testing, counseling, interview, ...",ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel
6228,PRODUCT_DESCRIPTION,YERSINIOSIS; Yersinia enterocolitica; psudotob...,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel


In [257]:
# Keeping the first appearance of question but dropping the rest
df_maven = df_maven.drop_duplicates(['QUESTION_ID'])

In [258]:
df_maven[df_maven.duplicated(['QUESTION_ID'])]

,QUESTION_ID,value,QUESTIONSET_ID,MODEL_NAME


In [259]:
df_maven = df_maven.reset_index(drop=True)

In [260]:
df_maven

,QUESTION_ID,value,QUESTIONSET_ID,MODEL_NAME
0,CURRENT_TELEPHONE_WORK,(999) 999-9999,DEMOGRAPHIC,DiseaseSurveillanceModel_General
1,EAT_YOGURT_PURCHASED_FROM,Yogurt Melts,RISK_HISTORY,DiseaseSurveillanceModel_General
2,CDR_PERSON_OR_AGENCY_REPORTING,Zyan Paresa,PORTAL_CDR_CASE_SUBMISSION,DiseaseSurveillanceModel_General
3,ANTACID_EXPOSURE,UNKNOWN,CLINICAL,DiseaseSurveillanceModel_General
4,DIARY_DAY_1_DINNER,Zippys -No Bean Chili,RISK_HISTORY,DiseaseSurveillanceModel_General
...,...,...,...,...
5439,COMMENTS,whatever,ADMINISTRATIVE,DiseaseSurveillanceModel_AggNETSS
5440,PRODUCT_CODE,YER,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel
5441,PROGRAM_CODE,STD,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel
5442,REPORT_CATEGORY_LAB,Urgent - upon order,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel


In [261]:
# Incidents elements
df_incidents = pd.read_csv('origami/csv/Incidents.csv',na_filter=False)
len(df_incidents)

1528

In [262]:
df_incidents.loc[:,'Label'].value_counts(dropna=False)

Label
                                       1083
CASE_REPORT_PRINT_COMPLETE                2
NETSS_ID                                  2
[INV155]Imported City                     2
What state does this case belong to       2
                                       ... 
Episode date                              1
Reciprocal notification_Event date        1
Illness End Date                          1
Diagnosis Date                            1
Insurance Plan                            1
Name: count, Length: 439, dtype: int64

In [263]:
# Get the two columns of interest and non-blank values
df_incidents = df_incidents[['Field','Label']]
df_incidents = df_incidents[df_incidents['Label']!='']
len(df_incidents)

445

In [264]:
df_incidents = df_incidents.reset_index(drop=True)

In [265]:
df_incidents

,Field,Label
0,IncidentID,Incident ID
1,ClientID,Client
2,IncidentTypeID,Incident Type
3,IncidentNumber,Incident Number
4,Status,Status
...,...,...
440,PolicySnapshotID,Policy Snapshot
441,LimitID,Limit
442,CatastropheID,Catastrophe
443,ClaimantInvolvedPartyID,Claimant Involved Party


In [266]:
# Original label to be used later for output
df_incidents.rename(columns={'Label':'original_label'},inplace=True)
df_incidents.columns

Index(['Field', 'original_label'], dtype='object')

In [267]:
# Remove only the first occurrence of [] in a string
def remove_squares(somestring):
    if ']' in somestring:
        for i in range(len(somestring)):
            if somestring[i]==']':
                return somestring[i+1:].strip()
    return somestring.strip()


In [268]:
remove_squares('asd[asdasd]asc[obas]da')

'asc[obas]da'

In [269]:
# Process the column values (eliminate the values in square brackets)
df_incidents.loc[:,'Label'] = [ remove_squares(label) for label in df_incidents.loc[:,'original_label']]

In [270]:
# Clean up column names in both data sets
df_incidents.loc[:,'Label'] =[ clean_up(q) for q in df_incidents.loc[:,'Label']]

In [271]:
df_incidents.loc[:,'Label'].value_counts()

Label
Marital Status                         2
NETSS_ID                               2
What state does this case belong to    2
Date Reported                          2
Imported City                          2
                                      ..
CDR_DATA_SUBMITTED_DATE                1
Case report date                       1
Date case was referred                 1
Date of inspection                     1
Insurance Plan                         1
Name: count, Length: 435, dtype: int64

In [272]:
# duplicates labels exist 
df_incidents[df_incidents.duplicated(['Label'],keep=False)]

,Field,original_label,Label
9,MaritalStatus,Marital Status,Marital Status
12,ReportDate,Date Reported,Date Reported
73,CustomText6,[INV165]MMWR Week,MMWR Week
74,CustomText7,[INV165_text]Override MMWR Week,Override MMWR Week
80,CustomText13,Reporting Source Type Code,Reporting Source Type Code
84,CustomText17,Reporting Source Type Code,Reporting Source Type Code
85,CustomText18,MMWR Week,MMWR Week
86,CustomText19,Override MMWR Week,Override MMWR Week
111,CustomCode14ID,Marital Status,Marital Status
112,CustomCode15ID,Race Category,Race Category


In [273]:
# dropping them
df_incidents = df_incidents.drop_duplicates(['Label'])

In [274]:
df_incidents[df_incidents.duplicated(['Label'],keep=False)]

,Field,original_label,Label


In [275]:
df_incidents = df_incidents.reset_index(drop=True)
df_incidents

,Field,original_label,Label
0,IncidentID,Incident ID,Incident ID
1,ClientID,Client,Client
2,IncidentTypeID,Incident Type,Incident Type
3,IncidentNumber,Incident Number,Incident Number
4,Status,Status,Status
...,...,...,...
430,PolicySnapshotID,Policy Snapshot,Policy Snapshot
431,LimitID,Limit,Limit
432,CatastropheID,Catastrophe,Catastrophe
433,ClaimantInvolvedPartyID,Claimant Involved Party,Claimant Involved Party


In [276]:
# Do a semantic match between Maven elements and Origami Incident labels

In [277]:
maven_elements = [str.replace(maven_de,'_',' ') for maven_de in df_maven.QUESTION_ID]
origami_elements = [str.replace(col,'_',' ') for col in df_incidents.Label]

In [278]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [279]:
# Compute embeddings
embeddings_maven = model.encode(maven_elements)
embeddings_origami = model.encode(origami_elements)

In [280]:
print(len(embeddings_origami))
print(len(embeddings_maven))
embeddings_origami
embeddings_maven

435
5444


array([[-0.16024935,  0.0271387 ,  0.06354488, ..., -0.00712903,
        -0.01673306,  0.06124594],
       [ 0.04245438,  0.03854417,  0.0684232 , ..., -0.06141916,
         0.08061311, -0.07027474],
       [-0.08168999,  0.03473966, -0.06668424, ..., -0.04572672,
        -0.00667135,  0.0398071 ],
       ...,
       [-0.06123391,  0.04124329, -0.09608547, ...,  0.11074385,
         0.03158536, -0.0520657 ],
       [ 0.03361172,  0.06674883, -0.07311527, ...,  0.01481686,
         0.04655252, -0.02631201],
       [-0.00503135,  0.02209355, -0.08806273, ..., -0.04621199,
         0.04993448,  0.04251292]], dtype=float32)

In [281]:
# Compute cosine similarities
similarities = model.similarity(embeddings_maven,embeddings_origami)

In [282]:
similarities

tensor([[ 0.0812,  0.1604,  0.0874,  ...,  0.0810,  0.1554,  0.1952],
        [ 0.0306,  0.0250,  0.0217,  ...,  0.0783, -0.0055,  0.0816],
        [ 0.2317,  0.2240,  0.1726,  ...,  0.0354,  0.3577,  0.1797],
        ...,
        [ 0.1367,  0.2358,  0.1663,  ...,  0.1559,  0.0138,  0.1511],
        [ 0.1265,  0.0612,  0.1655,  ...,  0.1260,  0.0043,  0.0017],
        [ 0.1315,  0.1426,  0.1554,  ...,  0.0672,  0.1814,  0.1052]])

In [283]:
print(type(similarities))
len(similarities)

<class 'torch.Tensor'>


5444

In [284]:
df_similarities = pd.DataFrame(similarities,index=df_maven.QUESTION_ID,columns=df_incidents.original_label)

In [285]:
df_similarities.to_csv('output/similarity_scores_Incidents.csv')

In [286]:
df_maven = df_maven.reset_index(drop=True)
df_maven.columns

Index(['QUESTION_ID', 'value', 'QUESTIONSET_ID', 'MODEL_NAME'], dtype='object')

In [287]:
df_output = df_maven[['MODEL_NAME','QUESTION_ID','QUESTIONSET_ID']]

In [288]:
len(df_output)

5444

In [289]:
#df_similarities.info()

In [290]:
#For every maven element (index), identify origami elements (column) that is the best match

filtered=[]
for idx in df_similarities.index:
    # Looking for columns with score higher than 0.75
    v = df_similarities.loc[idx,:]>0.75

    # Get the column with the maximum score
    max=0.0
    max_col=''
    for i,col in enumerate(df_similarities.columns):
        if v.iloc[i]==True:
            #print('value',df_similarities.loc[idx,col])
            if (df_similarities.loc[idx,col] > max):
                max = df_similarities.loc[idx,col]
                max_col=col

    #filtered.append({col:df_similarities.loc[idx,col]})
    #print(idx+','+max_col+':'+str(max)+'\n')
    filtered.append({'QUESTION_ID':idx,'target_label':max_col})



In [291]:
df_filtered = pd.DataFrame(filtered)
print(len(df_filtered))
#df_filtered.loc[:,'target_label'].value_counts(dropna=False)

5444


In [292]:
df_merge = pd.merge(df_filtered,df_incidents,left_on='target_label',right_on='original_label',how='left')

In [293]:
df_merge

,QUESTION_ID,target_label,Field,original_label,Label
0,CURRENT_TELEPHONE_WORK,,NaN,NaN,NaN
1,EAT_YOGURT_PURCHASED_FROM,,NaN,NaN,NaN
2,CDR_PERSON_OR_AGENCY_REPORTING,CDR_PERSON_OR_AGENCY_REPORTING,CustomText54,CDR_PERSON_OR_AGENCY_REPORTING,CDR_PERSON_OR_AGENCY_REPORTING
3,ANTACID_EXPOSURE,,NaN,NaN,NaN
4,DIARY_DAY_1_DINNER,,NaN,NaN,NaN
...,...,...,...,...,...
5439,COMMENTS,,NaN,NaN,NaN
5440,PRODUCT_CODE,,NaN,NaN,NaN
5441,PROGRAM_CODE,,NaN,NaN,NaN
5442,REPORT_CATEGORY_LAB,,NaN,NaN,NaN


In [294]:
# merge the above merged df with output dataframe
df_final = pd.merge(df_output,df_merge,on='QUESTION_ID')
#df_final

In [295]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5444 entries, 0 to 5443
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   MODEL_NAME      5444 non-null   category
 1   QUESTION_ID     5444 non-null   object  
 2   QUESTIONSET_ID  5444 non-null   object  
 3   target_label    5444 non-null   object  
 4   Field           273 non-null    object  
 5   original_label  273 non-null    object  
 6   Label           273 non-null    object  
dtypes: category(1), object(6)
memory usage: 261.0+ KB


In [296]:
# Add, rename, remove columns as needed
df_final['target_table'] = ['' if f=='' else 'Incidents' for f in df_final.target_label]
df_final=df_final.rename(columns={'Field':'target_field'})
df_final=df_final.drop('Label',axis=1)


In [297]:
df_final = df_final[["MODEL_NAME",'QUESTION_ID','target_table','target_field','target_label']]
#df_final

In [298]:
df_final

,MODEL_NAME,QUESTION_ID,target_table,target_field,target_label
0,DiseaseSurveillanceModel_General,CURRENT_TELEPHONE_WORK,,NaN,
1,DiseaseSurveillanceModel_General,EAT_YOGURT_PURCHASED_FROM,,NaN,
2,DiseaseSurveillanceModel_General,CDR_PERSON_OR_AGENCY_REPORTING,Incidents,CustomText54,CDR_PERSON_OR_AGENCY_REPORTING
3,DiseaseSurveillanceModel_General,ANTACID_EXPOSURE,,NaN,
4,DiseaseSurveillanceModel_General,DIARY_DAY_1_DINNER,,NaN,
...,...,...,...,...,...
5439,DiseaseSurveillanceModel_AggNETSS,COMMENTS,,NaN,
5440,IsolationAndControlModel,PRODUCT_CODE,,NaN,
5441,IsolationAndControlModel,PROGRAM_CODE,,NaN,
5442,IsolationAndControlModel,REPORT_CATEGORY_LAB,,NaN,


In [299]:
#df_final[df_final['target_field'].isna()]
df_final.loc[:,'MODEL_NAME'].value_counts(dropna=False)

MODEL_NAME
DiseaseSurveillanceModel_General        3192
DiseaseSurveillanceModel_STD            1221
Childhood_Lead_Child_Model               361
DiseaseSurveillanceModel_Hep             296
ClusterModel                             195
Childhood_Lead_Investigation_Model       125
PortalApplicationModel                    30
DiseaseSurveillanceModel_Coinfection      10
EHRAndScreeningModel_STD                   8
IsolationAndControlModel                   4
DiseaseSurveillanceModel_AggNETSS          2
Name: count, dtype: int64

In [300]:
# Convert category type back to str object
df_final.MODEL_NAME = df_final.MODEL_NAME.astype('str')

In [301]:
#df_final.info()

In [302]:
# Rename model names
df_final.loc[:,'MODEL_NAME'] = [ m.split('_')[1] if 'DiseaseSurveillanceModel_' in m else m for m in df_final.loc[:,'MODEL_NAME']]

In [303]:
df_final.loc[:,'MODEL_NAME'].value_counts(dropna=False)

MODEL_NAME
General                               3192
STD                                   1221
Childhood_Lead_Child_Model             361
Hep                                    296
ClusterModel                           195
Childhood_Lead_Investigation_Model     125
PortalApplicationModel                  30
Coinfection                             10
EHRAndScreeningModel_STD                 8
IsolationAndControlModel                 4
AggNETSS                                 2
Name: count, dtype: int64

In [304]:
df_final.to_csv('output/maven_Incidents_origami_similarity_match.csv',index=False)